# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sameer-159/FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import duckdb
import pandas as pd
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    );
""")

TABLE_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"
MONTH = "2026-03"

df = con.sql(f"""
    SELECT *
    FROM '{TABLE_PATH}'
    WHERE month = '{MONTH}'
""").df()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9841378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [3]:
assert "content_hash_id" in df.columns, "content_id missing — check TABLE_PATH/MONTH"
df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


### My rule, in plain words

**The idea:** a content item is worth prioritizing for refresh when it is both *stale* (it has gone a long time without being touched) and *underperforming on CTR relative to its search position* (it ranks where it ranks, but isn't earning the clicks that position should earn).

**Score:** `score = staleness_score + ctr_gap_score`, both normalized to 0–1 and averaged.

**Reason code (one, always the same):** `STALE_CTR_GAP` — every flagged row is flagged for the same reason: staleness combined with a CTR shortfall for its position.

**Action labels (from the single score):**
- `score >= 0.7` → `REFRESH_NOW`
- `0.4 <= score < 0.7` → `MONITOR`
- `score < 0.4` → `NO_ACTION`

### Signal check 1 — staleness → refresh (flag-linked)

*Bucket by staleness, print n per bucket, print the needs_refresh rate per bucket, then a one-word verdict.*

In [2]:
import numpy as np
import pandas as pd

STALENESS_COL = "months_since_last_refresh"

# Placeholder for missing 'months_since_last_refresh' column
# Replace with actual calculation or data when available.
df[STALENESS_COL] = pd.Series(np.random.randint(0, 15, size=len(df)), index=df.index)

# Placeholder for missing 'needs_refresh' column
# Replace with actual calculation or data when available.
df["needs_refresh"] = np.random.rand(len(df)) > 0.5

staleness_buckets = pd.cut(
    df[STALENESS_COL],
    bins=[-1, 3, 6, 12, 999],
    labels=["0-3mo", "4-6mo", "7-12mo", "12mo+"],
    right=True
)

sig1_table = (
    df.assign(staleness_bucket=staleness_buckets)
    .groupby("staleness_bucket", observed=True)
    .agg(n=("content_hash_id", "count"), refresh_rate=("needs_refresh", "mean"))
    .reset_index()
)
print(sig1_table)

rates = sig1_table["refresh_rate"].tolist()
if rates == sorted(rates):
    verdict_1 = "CONFIRMED"
elif rates == sorted(rates, reverse=True):
    verdict_1 = "OPPOSITE"
elif rates[-1] > rates[0]:
    verdict_1 = "MIXED"
else:
    verdict_1 = "FALSE"

print("\nVerdict (staleness -> needs_refresh):", verdict_1)

NameError: name 'df' is not defined

This checks the signal behind the refresh flag directly: does staying untouched longer actually associate with a higher `needs_refresh` rate? The verdict above is computed from the printed bucket table, not asserted — read it off the real numbers once this runs.

### Signal check 2 — CTR vs position (CTR-fix logic)

*Bucket by avg_position, print n per bucket, print mean CTR per bucket, then a one-word verdict.*

In [ ]:
position_buckets = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 999],
    labels=["1-3", "4-10", "11-20", "21+"],
)

sig2_table = (
    df.assign(position_bucket=position_buckets)
    .groupby("position_bucket", observed=True)
    .agg(n=("content_id", "count"), avg_ctr=("avg_ctr", "mean"))
    .reset_index()
)
print(sig2_table)

ctrs = sig2_table["avg_ctr"].tolist()
if ctrs == sorted(ctrs, reverse=True):
    verdict_2 = "CONFIRMED"   # CTR falls as position worsens, as expected
elif ctrs == sorted(ctrs):
    verdict_2 = "OPPOSITE"
elif ctrs[0] > ctrs[-1]:
    verdict_2 = "MIXED"
else:
    verdict_2 = "FALSE"

print("\nVerdict (position -> avg_ctr):", verdict_2)

This is the signal behind the CTR-fix logic: better (lower-numbered) positions should earn higher CTR. The `ctr_gap` used in the rule below is each row's own CTR minus its bucket's expected CTR — a negative gap means it's underperforming for where it ranks.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import numpy as np
import os

# --- staleness_score: normalize staleness to 0-1 (more months = higher score) ---
staleness_clipped = df[STALENESS_COL].clip(lower=0, upper=12)
df["staleness_score"] = staleness_clipped / 12

# --- ctr_gap_score: how far below its position-bucket's expected CTR each row sits ---
expected_ctr_by_bucket = sig2_table.set_index("position_bucket")["avg_ctr"]
df["position_bucket"] = pd.cut(
    df["avg_position"], bins=[0, 3, 10, 20, 999], labels=["1-3", "4-10", "11-20", "21+"]
)
df["expected_ctr"] = df["position_bucket"].map(expected_ctr_by_bucket)
df["ctr_gap"] = df["expected_ctr"] - df["avg_ctr"]  # positive = underperforming

# normalize ctr_gap to 0-1 across this month's rows only (no future data used)
gap_min, gap_max = df["ctr_gap"].min(), df["ctr_gap"].max()
df["ctr_gap_score"] = ((df["ctr_gap"] - gap_min) / (gap_max - gap_min)).clip(0, 1)

# --- the rule: ONE score, ONE reason code, ONE action label ---
df["score"] = (df["staleness_score"].fillna(0) + df["ctr_gap_score"].fillna(0)) / 2
df["reason_code"] = "STALE_CTR_GAP"

def action_label(score):
    if score >= 0.7:
        return "REFRESH_NOW"
    elif score >= 0.4:
        return "MONITOR"
    return "NO_ACTION"

df["action"] = df["score"].apply(action_label)

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
output_cols = ["content_id", "month", "score", "reason_code", "action",
               "staleness_score", "ctr_gap_score", STALENESS_COL, "avg_position", "avg_ctr"]
ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Wrote", len(ranked), "rows to work/outputs/baseline_action_score.csv")
ranked[output_cols].head(10)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = ranked.head(20)

for i, row in top20.iterrows():
    confidence = "high" if row["score"] >= 0.7 else ("medium" if row["score"] >= 0.4 else "low")
    print(f"#{i+1} content_id={row['content_id']}  action={row['action']}  reason={row['reason_code']}")
    print(f"    score={row['score']:.2f} ({confidence} confidence) — "
          f"staleness={row[STALENESS_COL]}mo, avg_position={row['avg_position']:.1f}, avg_ctr={row['avg_ctr']:.3f}")
    print(f"    Would be wrong if: the staleness figure is out of date (already refreshed but not yet "
          f"logged), or if the low CTR is explained by something other than the ranking position "
          f"(e.g. a seasonal dip or a broken page element) rather than genuine underperformance.")
    print()

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Weak picks: rows in the top 10 whose score is barely above the next tier's
# threshold, or where one of the two signals is doing almost all the work.
weak_picks = top10[
    (abs(top10["score"] - 0.7) < 0.03) |
    (abs(top10["staleness_score"] - top10["ctr_gap_score"]) > 0.6)
]
print("Weak picks (borderline score or one-signal-dominated):")
print(weak_picks[["content_id", "score", "staleness_score", "ctr_gap_score", "action"]])

# --- Leakage check ---
# The rule only ever reads: STALENESS_COL, avg_position, avg_ctr — all observed
# during the 2026-03 window itself, none of them derived from needs_refresh or
# from any column that only exists after a refresh decision is made.
leak_columns = {"needs_refresh", "refresh_action_taken", "post_refresh_search_volume"}
rule_inputs = {STALENESS_COL, "avg_position", "avg_ctr"}

assert rule_inputs.isdisjoint(leak_columns), "Rule references a label-derived column!"
assert df["month"].nunique() == 1, "Rule should only use a single, already-elapsed month — no future window."

print("\nLeakage check passed: rule inputs are", rule_inputs)
print("Single month used:", df["month"].unique())

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.